# Why the Multi-View Posterior is Strictly Tighter than Any Single-View Posterior

Context: in `nf_merging.py`, the `sample_param_avg` baseline uses the per-view posterior mean of NF samples and (per-view) approximates $\mathbb{E}[\beta \mid I_i]$. Tempered IS approximates the multi-view posterior mean $\mathbb{E}[\beta \mid I_1,\dots,I_V]$. This note formalises why the latter has strictly lower expected $L^2$ error.

## 1. Setup

Under $L^2$ loss, the Bayes-optimal estimator of $\beta$ given some data $D$ is the posterior mean $\mu_D = \mathbb{E}[\beta \mid D]$, and its expected error equals the (averaged) posterior variance:

$$
\mathbb{E}_{\beta, D}\bigl[(\beta - \mu_D)^2\bigr] \;=\; \mathbb{E}_D\bigl[\operatorname{Var}(\beta \mid D)\bigr].
$$

So comparing Bayes estimators reduces to comparing posterior variances. The claim becomes

$$
\mathbb{E}\bigl[\operatorname{Var}(\beta \mid I_1,\dots,I_V)\bigr] \;\le\; \mathbb{E}\bigl[\operatorname{Var}(\beta \mid I_i)\bigr] \qquad \text{for every } i.
$$

## 2. Distribution-free proof (law of total variance)

For any random variables $\beta, X, Y$, the law of total variance, applied **pointwise in $X$**, gives

$$
\operatorname{Var}(\beta \mid X) \;=\; \mathbb{E}\bigl[\operatorname{Var}(\beta \mid X, Y) \,\big|\, X\bigr] \;+\; \operatorname{Var}\bigl(\mathbb{E}[\beta \mid X, Y] \,\big|\, X\bigr) \;\;\ge\;\; \mathbb{E}\bigl[\operatorname{Var}(\beta \mid X, Y) \,\big|\, X\bigr].
$$

Set $X = I_i$ and $Y = \{I_j\}_{j\ne i}$, then take expectation over $I_i$:

$$
\mathbb{E}\bigl[\operatorname{Var}(\beta \mid I_i)\bigr] \;\ge\; \mathbb{E}\bigl[\operatorname{Var}(\beta \mid I_1,\dots,I_V)\bigr].
$$

Equality holds iff $\mathbb{E}[\beta \mid I_1,\dots,I_V] = \mathbb{E}[\beta \mid I_i]$ almost surely — i.e. the other views carry zero information about $\beta$ beyond what $I_i$ already tells you. For any non-degenerate multi-view setup (distinct camera angles, distinct occlusion patterns), this fails and the inequality is strict.

**This argument makes no Gaussianity or independence assumption** — it is a pure consequence of the tower rule. It holds for the NF posterior, even though the NF is non-Gaussian.

## 3. Gaussian case for intuition

When $\beta \sim \mathcal{N}(0, \Sigma_0)$ and $I_i = H_i \beta + \varepsilon_i,\;\varepsilon_i \sim \mathcal{N}(0, \Sigma_i)$ with conditionally independent $\varepsilon_i$, the posterior is Gaussian with precision

$$
\Lambda_{\mathrm{full}} \;=\; \Sigma_0^{-1} \;+\; \sum_{i=1}^{V} H_i^{\top}\, \Sigma_i^{-1}\, H_i.
$$

Each $H_i^{\top} \Sigma_i^{-1} H_i$ is positive semi-definite, so adding views can only **add precision**:

$$
\Lambda_{\mathrm{full}} \;\succeq\; \Lambda_{S} \;\succeq\; \Lambda_{\{i\}} \qquad \text{(Loewner order)}.
$$

Equivalently, posterior covariance shrinks in **every direction** simultaneously: for any unit vector $v \in \mathbb{R}^{D}$,

$$
v^{\top}\, \Sigma_{\mathrm{full}}\, v \;\le\; v^{\top}\, \Sigma_i\, v.
$$

There is no direction in $\beta$-space along which adding views can widen the posterior. The strict-versus-equal question reduces to whether $H_j^{\top} \Sigma_j^{-1} H_j$ has any component in the row span of $v$ — if any added view projects onto direction $v$ even weakly, the variance along $v$ strictly drops.

## 4. Why it is strict for body shape from multi-view images

Each view's likelihood projects $\beta$ through a view-dependent observation operator (camera ray, silhouette, 2D-keypoint reprojection). The null space of $H_i$ is roughly the set of shape changes that produce the same image from view $i$ — predominantly **the depth-aligned shape components for that view**.

- Frontal view: $H_{\mathrm{frontal}}$ is rank-deficient along front–back-depth shape axes. $\operatorname{Var}(\beta_{\mathrm{depth}} \mid I_{\mathrm{frontal}})$ is large.
- Side view: rank-deficient along left–right axes. $\operatorname{Var}(\beta_{\mathrm{lateral}} \mid I_{\mathrm{side}})$ is large.

Their null spaces intersect only in the genuinely-unobservable subspace (e.g. global scale $\times$ camera distance, which the IS merge does not try to resolve). Outside that intersection, **every view tightens at least one direction the others do not**, so

$$
\Lambda_{\mathrm{full}} \;\succ\; \Lambda_{\{\text{any single view}\}}
$$

strictly, in the directions where other views are uncertain.

## 5. Implication for the comparison

`sample_param_avg`'s reported metric is, in expectation, lower-bounded by the **single-view** posterior variance — that is the floor for any estimator that only uses one view. The multi-view posterior mean has expected error equal to the **joint** posterior variance, which is strictly smaller in every direction where views are complementary. Tempered IS attempts to approximate this joint posterior mean across $V \cdot S$ candidates; even with the tempering bias, the gap between $\operatorname{Var}(\beta \mid I_i)$ and $\operatorname{Var}(\beta \mid I_1,\dots,I_V)$ is what gives it room to beat `sample_param_avg`.

The same argument shows you cannot beat `sample_param_avg` by picking the single most-informative view — single-view posteriors are bounded below by their own posterior variance, which is strictly larger than the joint one in every direction where any other view contributes.

# Why Tempered-Merged Can Lose to $\min_i \texttt{sample\_param\_avg}_i$ in Practice

Section 2 says the **Bayes-optimal** multi-view posterior mean has lower **expected** $L^2$ error than the **Bayes-optimal** single-view posterior mean. Three things break that guarantee in practice; failures usually compound.

## 6. Failure mode A — neither side of the theorem holds with a learned flow

The flow `flow_beta` learns an approximation $\hat p(\beta \mid I)$ of the true posterior. The cross-view product becomes

$$
\hat p_{\mathrm{joint}}(\beta) \;\propto\; \prod_j \hat p\bigl(\beta - \mu_j \,\big|\, c_j\bigr),
$$

evaluated on residuals against the **deterministic regressor's per-view means** $\mu_j$ — not the true conditional means. This opens three failure modes:

1. **Mis-calibrated stage-1 flow.** If $\hat p(\cdot \mid c)$ is overconfident in the depth-ambiguous direction, the log-likelihood ratio across views is artificially sharp — cross-view product locks onto whatever residual is "least surprising" under all $V$ flows simultaneously, dominated by the *modes* of each $\hat p$, not their true posteriors. Underconfident is the opposite: weights flatten and the merge degenerates to a uniform $V \cdot S$ average.
2. **Biased per-view mean predictions.** If $\mu_j$ has correlated bias across views (e.g. the mean head systematically underestimates body height), every $\hat p(\beta - \mu_j \mid c_j)$ peaks at a $\beta$ that inherits that bias. The consensus $\beta^*$ is the joint MAP of $V$ similarly-biased flows — bias does not average out the way variance does. Per-view `sample_param_avg` inherits the same bias, but `min_i` *selects* the realisation where the bias happens to be smallest.
3. **2D-keypoint consistency loss in training.** Per `loss_kp2d_samples`, NF samples are pulled toward 2D-consistent reprojections during training, not pure NLL. The flow's high-likelihood region is biased toward 2D-consistent shapes for view $j$'s detected keypoints. Cross-view product over $V$ such biased flows compounds, not cancels, that bias when 2D keypoints have correlated noise (same person, same lighting/clothing across views).

## 7. Failure mode B — tempering at fixed $T = 15$ is a wrong-on-average estimator

The tempered IS estimate is

$$
\hat \beta^{\mathrm{temp}} \;=\; \mathbb{E}_{p^{1/T}}[\beta] \;\ne\; \mathbb{E}_{p}[\beta].
$$

For $p$ skewed or heavy-tailed (as a learned NF often is), the $p^{1/T}$ mean is biased relative to the true posterior mean — and the bias direction depends on flow asymmetry, not on view geometry. With $T=15$ hardcoded (rather than the principled $T=D=55$), the choice is empirically tuned for one regime and may go wrong elsewhere.

The other concrete failure is that tempering may not flatten enough:

- If the spread of $\log w$ across the $V \cdot S$ candidates exceeds $T \log(V \cdot S) \approx 15 \cdot 7$, softmax stays near one-hot. Diagnostic: compute effective sample size

$$
\mathrm{ESS} \;=\; \frac{1}{\sum_k w_k^2}.
$$

If $\mathrm{ESS} \ll V \cdot S$ (say, $\mathrm{ESS}$ in the single digits), the merge is closer to "pick one sample" than "average many." `sample_param_avg` has $\mathrm{ESS} = S$ within view by construction and averages in **parameter space** before MHR (Jensen-friendly here), so it gets the variance reduction. Tempered IS may be using fewer effective samples → noisier estimate.

## 8. Failure mode C — the comparison itself favours the oracle

$\min_i \texttt{sample\_param\_avg\_pvetsc}_i$ is an order statistic over $V$ random realisations. For $V$ views with per-view error variance $\sigma^2$, the expected minimum scales like

$$
\mathbb{E}\bigl[\min_i |X_i|\bigr] \;\sim\; \sigma \cdot \Phi^{-1}(1/V),
$$

(sub-linear in $V$) — better than $\sigma$ from any single view, and better than $\sigma / \sqrt{V}$ from perfect averaging when $V$ is small.

Back-of-envelope: with $V = 4$ Gaussian-like view errors of std $\sigma$,

$$
\mathbb{E}[\min_i |X_i|] \approx 0.45\, \sigma, \qquad \mathrm{std}\bigl(\tfrac{1}{V} \sum_i X_i\bigr) = 0.5\, \sigma.
$$

So **even a perfect Bayesian fuser would lose to oracle-min at small $V$** unless cross-view information is genuinely complementary (i.e. $H_i$ have meaningfully different null spaces). For nearly-redundant views, oracle-min beats fusion because the theorem only promises improvement over a *single fixed* view, not over a max-of-$V$-realisations.

## 9. Diagnostics — what to actually check in a failing run

In rough order of explanatory power:

1. **ESS of the tempered weights.** `ess_b = 1.0 / (is_weights[b].pow(2).sum())` after pooling across $V \cdot S$. If median ESS is $\ll V \cdot S$ (e.g. ESS $< 10$ at $V \cdot S = 400$), you are sample-collapsed — increase $T$ until ESS $\sim 0.3 \cdot V \cdot S$ and re-evaluate.
2. **Is `sample_param_avg` (per-view, average over $V$) already $\le$ tempered merged?** If so, fusion is genuinely losing ground, not just losing the order-statistic battle. Points to flow miscalibration / mean-prediction bias (mode A1, A2), not the comparison (mode C).
3. **Replace `mean_beta_j` with `sample_param_avg_j` in the tempered merge.** If the per-view mean predictor is biased relative to the per-view posterior mean, evaluating residuals against the latter (which the flow saw at training) is the cleaner reference. Significant change → mode A2 is real.
4. **Compare tempered merged vs uniform $V \cdot S$ average.** If tempered $\approx$ uniform, cross-view weighting is doing nothing useful — the flow is not separating signal from noise, which is mode A1.
5. **Compare tempered merged vs Gaussian precision-weighted merge** (`merge_params_nf_gaussian`). If Gaussian beats tempered, the NF's higher-order structure is *hurting* — likely A1 (overconfidence) or A3 (2D-keypoint training contamination).

## 10. Summary

Bayesian fusion strictly dominates a single fixed view only with a calibrated joint posterior. With

- a learned posterior approximation,
- biased deterministic per-view means used as residual references,
- tempering with a fixed $T$ that is wrong-on-average,
- evaluated against an order-statistic oracle baseline ($\min_i$ over $V$ realisations),

losing to oracle-min is a normal outcome. Most of the diagnostic work is figuring out which of A/B/C is dominant in a given run — only A is fixable by retraining; B is fixable by rethinking the temperature schedule; C is a property of the chosen baseline and cannot be 'fixed', only bounded with theory.

## 11. Does the flow's residual modelling rescue tempered IS from A2?

A natural hope: even if the regressor's per-view mean $\mu_j$ is biased, the flow is trained on **residuals** $\Delta\beta_{gt} = \beta_{gt} - \mu_j$, so it should learn to absorb that bias and put uncertainty around the corrected location. Does this make beating $\min_i \texttt{sample\_param\_avg}_i$ easy again?

**Partly — but not enough.** Three remaining obstacles persist even when the flow's mean is exact.

### 11.1 What the flow can correct

If the regressor has bias $b_j = \mathbb{E}[\beta_{gt} \mid I_j] - \mu_j$, NLL training pushes the flow's residual mean toward the same $b_j$:

$$
\mathbb{E}_{\hat p}[\Delta\beta \mid c_j] \;\approx\; \mathbb{E}[\beta_{gt} - \mu_j \mid I_j] \;=\; b_j.
$$

So the per-view posterior mean estimator

$$
\hat\mu_j^{\mathrm{full}} \;=\; \mu_j \;+\; \mathbb{E}_{\hat p}[\Delta\beta \mid c_j] \;\approx\; \mathbb{E}[\beta_{gt} \mid I_j]
$$

is approximately unbiased. `sample_param_avg_j` is exactly this estimator (Monte-Carlo over $S$ samples). So the user's hope **already holds for `sample_param_avg`** — including the oracle baseline $\min_i \texttt{sample\_param\_avg}_i$. The relevant question is therefore: does that same correction propagate into tempered IS?

### 11.2 Tempered IS demands more than a correct mean

Tempered IS evaluates, for a candidate $\beta_i^k$ from view $i$ scored under view $j$,

$$
\log \hat p\bigl(\beta_i^k - \mu_j \,\big|\, c_j\bigr).
$$

For this product to recover the true joint posterior, the flow must approximately match the **entire conditional density**, not just its mean:

1. **Variance / tail shape.** If $\hat p(\cdot \mid c_j)$ is overconfident in the depth-ambiguous direction, even a perfectly bias-corrected mean is useless: cross-view product applies very large negative log-likelihoods to candidates that should still be plausible. The joint MAP/mean lands at the intersection of $V$ over-narrow ellipsoids — sharply concentrated but at the wrong location.
2. **Off-diagonal structure.** The flow couples shape components. If learned correlations are wrong, residual directions that should be cheap are penalised, skewing the consensus.
3. **Bias amplification through products.** Even when each $\hat p(\beta \mid I_j)$ has unbiased mean and roughly correct variance, the joint mean

$$
\beta^* \;\approx\; \biggl(\sum_j \Lambda_j\biggr)^{-1} \sum_j \Lambda_j \bigl(\mu_j + b_j\bigr)
$$

amplifies any **direction-correlated residual bias** by the joint precision. A small per-view location error becomes the joint location error, while joint posterior variance shrinks as $1/V$ — so the **bias-to-noise ratio of the merged estimate is $V\times$ larger than per view**. `sample_param_avg` only inherits one $b_j$; the joint inherits a precision-weighted sum of all of them, with no cancellation if biases are correlated across views.

### 11.3 Order-statistic gap remains

Even granting that the flow corrects the regressor mean perfectly (so $\hat\mu_j^{\mathrm{full}}$ is exactly unbiased), the comparison is

$$
\underbrace{\min_i \bigl| \hat\mu_i^{\mathrm{full}} - \beta_{gt} \bigr|}_{\text{oracle}} \quad \text{vs.} \quad \underbrace{\bigl| \hat\beta^{\mathrm{temp}} - \beta_{gt} \bigr|}_{\text{tempered}}.
$$

The oracle is an order statistic over $V$ realisations of an estimator with residual variance $\sigma^2$. Its expected value scales like $\sigma \cdot \Phi^{-1}(1/V)$, sub-linear in $V$. The fused estimate has variance $\sigma^2 / V$ at best (perfect averaging), so std $\sigma / \sqrt{V}$. For $V = 4$:

$$
\mathbb{E}[\min_i |X_i|] \approx 0.45\, \sigma \quad \text{vs.} \quad \sigma / \sqrt{4} = 0.50\, \sigma.
$$

So **even with a perfectly bias-corrected flow, oracle-min still wins at small $V$ in expectation**, because the fuser is one realisation while the oracle gets to pick the lucky one of $V$. Bias correction shifts both estimators equally — it does not change the sub-linear-vs-$\sqrt{V}$ scaling that produces the gap.

### 11.4 Where flow correction does help

The hope is real for two regimes:

- **Large $V$.** $\Phi^{-1}(1/V)$ flattens, $1/\sqrt{V}$ does not — fusion eventually wins. With $V \ge 16$ or so, perfect fusion clearly beats oracle-min.
- **Complementary views (different $H_j$ null spaces).** The joint posterior variance $\sigma_{\mathrm{joint}}^2$ is then much smaller than $\sigma^2 / V$ in the ambiguous directions of each individual view — fusion's per-direction gain becomes large enough to dominate the order-statistic effect even at small $V$.

### 11.5 Net answer

Yes, the flow's residual modelling absorbs the deterministic regressor's mean bias, and that *is* what makes tempered IS competitive at all. But the remaining difficulty in beating oracle-min comes from three sources that bias correction does not touch:

1. **flow shape/tail miscalibration**, which the cross-view product is much more sensitive to than the per-view mean;
2. **bias amplification of cross-view multiplication** when per-view biases are correlated (not fixed by per-view bias correction);
3. **the structural advantage of taking $\min$ over $V$ realisations**, independent of estimator unbiasedness.

Even with the flow's mean exactly correct, tempered IS still has to win on a $\sqrt{V}$ vs $\Phi^{-1}(1/V)$ comparison, plus survive the multiplicative shape sensitivity that doesn't bite `sample_param_avg`. Bias correction makes the fight winnable in principle; it does not make it easy.

# Best-Case Fusion: Strongly Complementary Crops

Section 8 showed that fusion has a hard time beating oracle-min when views are roughly redundant. The opposite extreme — **strongly complementary** views, e.g. one crop showing only the upper body and another only the lower body — is the **best-case scenario** for fusion. It cleanly defeats every failure mode in sections 6–8 in theory, and gives a sharp diagnostic test for the implementation in practice.

## 12. Per-view posterior structure under extreme crops

With an upper-body crop, view 1's observation operator $H_1$ has support **only** in upper-body shape components; lower-body components live entirely in the null space. So

$$
\operatorname{Var}\bigl(\beta_{\mathrm{upper}} \mid I_1\bigr) \;\approx\; \sigma_{\mathrm{obs}}^{2} \quad \text{(small, well-constrained)},
$$

$$
\operatorname{Var}\bigl(\beta_{\mathrm{lower}} \mid I_1\bigr) \;\approx\; \sigma_{\mathrm{prior}}^{2} \quad \text{(large, prior-level uncertainty)},
$$

and symmetric for view 2. The joint posterior is well-constrained in both halves:

$$
\operatorname{Var}(\beta_{\mathrm{upper}} \mid I_1, I_2) \;\approx\; \sigma_{\mathrm{obs}}^{2}, \qquad \operatorname{Var}(\beta_{\mathrm{lower}} \mid I_1, I_2) \;\approx\; \sigma_{\mathrm{obs}}^{2}.
$$

The two views' null spaces are **nearly orthogonal**: the upper crop is uncertain exactly where the lower crop is certain, and vice versa.

## 13. The fusion gap dominates the order-statistic gap

Per-view error magnitude (averaged across components):

$$
\sigma_{\mathrm{per-view}}^{2} \;\approx\; \tfrac{1}{2}\sigma_{\mathrm{prior}}^{2} \;+\; \tfrac{1}{2}\sigma_{\mathrm{obs}}^{2} \;\approx\; \tfrac{1}{2}\sigma_{\mathrm{prior}}^{2}, \qquad \text{since } \sigma_{\mathrm{prior}} \gg \sigma_{\mathrm{obs}}.
$$

Joint error magnitude:

$$
\sigma_{\mathrm{joint}}^{2} \;\approx\; \sigma_{\mathrm{obs}}^{2}.
$$

Fusion's gain factor is

$$
\frac{\sigma_{\mathrm{per-view}}}{\sigma_{\mathrm{joint}}} \;=\; \mathcal{O}\!\left(\frac{\sigma_{\mathrm{prior}}}{\sigma_{\mathrm{obs}}}\right),
$$

potentially $10\times$ or $100\times$. The order-statistic factor of $\Phi^{-1}(1/V) / (1/\sqrt{V})$ is $\mathcal{O}(1)$. Fusion wins by orders of magnitude.

The deeper reason: **oracle-min cannot close this gap**. The min over $V$ partial estimates is bounded below by the prior-level uncertainty in whichever half each view doesn't see. Fusion is not bounded by this — it observes both halves simultaneously through the cross-view product.

## 14. Conditions for the implementation to deliver

The framework predicts a clean win, but two things must hold for the code to actually realise it:

1. **The flow must inflate its variance correctly for unobserved components.** If trained only on full-body images, the flow has never seen contexts where some shape components are unconstrained — it will likely be **overconfident** about lower-body parameters when given an upper-body crop, regressing toward the population mean with implausibly small NF variance. Cross-view product of two overconfident-but-wrong-direction posteriors gives a sharp but biased joint. **Fix: include extreme crops in training data.**
2. **The per-view mean predictor $\mu_j$ should output the prior mean for unobserved components.** If it instead outputs an arbitrary "best guess" with overconfident NF variance, the same compounding occurs. The flow training would need to see crop-conditional uncertainty so its residual variance learns to widen for unobserved dims.

## 15. What this scenario lets you test

Extreme cropping is also a clean experimental probe: it **isolates the fusion gain from view redundancy**. If your tempered merge cannot beat oracle-min on upper-half / lower-half crops of the same subject, the failure is not failure mode C (order statistic) — it is failure mode A1 (flow miscalibration on out-of-distribution unconstrained components).

This narrows the diagnosis: passing this test tells you the cross-view IS pipeline is sound; failing it tells you the flow's per-view uncertainty is not crop-aware. Either way, it is a more decisive experiment than evaluating on full-body multi-view, where view redundancy lets the order-statistic baseline stay close.

# When Diverse Per-View Samples Are Not Enough: IS Proposal–Target Mismatch

**Empirical observation.** A model trained with extreme-cropping data produces NF samples that are visibly diverse in the unobserved regions (so failure mode A1 — overconfidence on out-of-distribution contexts — is ruled out). And yet, tempered IS still fails to beat $\min_i \texttt{sample\_param\_avg}_i$.

This is a strong negative result. It rules out the simplest hypothesis (model miscalibration on unconstrained components) and points to a **methodological** failure: high-dimensional importance sampling cannot recover the joint posterior of strongly complementary views, even with perfect per-view marginals.

## 16. The proposal–target mismatch argument

Consider scoring a view-1 (upper crop) candidate $\beta_1^k$ under view-2 (lower crop)'s flow. Decompose by half:

- **Upper components of $\beta_1^k$**: tight (view 1 saw them). View 2 has no signal here, so $\hat p(\beta_1^k - \mu_2 \mid c_2)$ is near-flat over upper. Fine — no penalty, no sharpening.
- **Lower components of $\beta_1^k$**: drawn from view 1's diverse samples (variance $\sim \sigma_{\mathrm{prior}}^2$). View 2 has a *tight* posterior here (variance $\sim \sigma_{\mathrm{obs}}^2$).

The probability that any single $\beta_1^k$'s lower components fall within view 2's tight posterior is roughly

$$
\biggl(\frac{\sigma_{\mathrm{obs}}}{\sigma_{\mathrm{prior}}}\biggr)^{d_{\mathrm{lower}}}.
$$

For body shape with $d_{\mathrm{lower}} \sim 22$ and $\sigma_{\mathrm{obs}} / \sigma_{\mathrm{prior}} \sim 0.1$, this is $\sim 10^{-22}$. With $S = 100$ samples per view, you draw zero candidates that lie in the joint-high-density region.

The cross-view log-weights are therefore dominated by candidates that are wrong in **both** halves to varying degrees — none are right in both. After softmax (tempered or not), the merged estimate is approximately a uniform average over $V \cdot S$ candidates, none of which sit near the joint posterior mode. That uniform average is roughly the average of `sample_param_avg_1` and `sample_param_avg_2`, which is *worse* than either individually for the half each view does see well.

This is the classical curse of dimensionality for IS: **IS works only when the proposal overlaps the target**. When views are strongly complementary, each view's proposal is wide exactly where the joint target is thin, so coverage of the joint mode is exponentially small in the unobserved-dimension count.

Tempering does not fix this — it only flattens the weights of candidates the proposals do produce; it cannot synthesise candidates in the joint-mode region.

### 16.1 The "thick slab" picture

In the 55-D shape+scale space:

- **View 1's posterior $\hat p(\beta \mid I_1)$** has its mass on a **thick slab** — thin (variance $\sigma_{\mathrm{obs}}^2$) along the upper-body axes that view 1 actually constrained, thick (variance $\sigma_{\mathrm{prior}}^2$) along the lower-body axes that view 1 has no information about. Geometrically: a $22$-dim-thick pancake in $55$-D, oriented along the unobserved-by-view-1 subspace.
- **View 2's posterior $\hat p(\beta \mid I_2)$** is the **orthogonal slab**: thin along the lower-body axes, thick along the upper-body axes — the same pancake shape, rotated $90^{\circ}$.
- **The joint posterior** $\hat p(\beta \mid I_1, I_2) \propto \hat p_1 \cdot \hat p_2$ is the **intersection** of the two slabs: a small ball around the truth, thin in both directions because each thinness is contributed by exactly one view.

When you sample from view 1, you fill out view 1's pancake — most of which lies *outside* the small intersection ball. The chance of any sample landing in the ball is the volume ratio of (intersection ball) to (view 1's pancake):

$$
\frac{\sigma_{\mathrm{obs}}^{d_{\mathrm{lower}}}}{\sigma_{\mathrm{prior}}^{d_{\mathrm{lower}}}} \;=\; \biggl(\frac{\sigma_{\mathrm{obs}}}{\sigma_{\mathrm{prior}}}\biggr)^{d_{\mathrm{lower}}}.
$$

Exponential decay in the unobserved-dimension count $d_{\mathrm{lower}}$ — this is the curse of dimensionality manifesting as a coverage failure.

### 16.2 What the cross-view weight actually computes

Decompose the candidate $\beta_1^k$ into halves: $\beta_1^k = (\beta_{1,\mathrm{up}}^k,\, \beta_{1,\mathrm{lo}}^k)$, and similarly $\mu_2 = (\mu_{2,\mathrm{up}},\, \mu_{2,\mathrm{lo}})$. The cross-view log-weight factors as

$$
\log w_1^k \;=\; \log \hat p(\beta_1^k - \mu_2 \mid c_2) \;\approx\; \underbrace{\log \hat p_{\mathrm{up}}(\beta_{1,\mathrm{up}}^k - \mu_{2,\mathrm{up}} \mid c_2)}_{\text{nearly flat: view 2 is loose on upper}} \;+\; \underbrace{\log \hat p_{\mathrm{lo}}(\beta_{1,\mathrm{lo}}^k - \mu_{2,\mathrm{lo}} \mid c_2)}_{\text{sharply peaked: view 2 is tight on lower}}.
$$

The **upper term is uninformative** — view 2's posterior is wide in the upper subspace, so almost any $\beta_{1,\mathrm{up}}^k$ gets roughly the same likelihood under view 2. View 2 cannot validate or reject the upper part of view 1's candidate; it has no opinion.

The **lower term is the entire signal** — view 2 has a sharp opinion about $\beta_{\mathrm{lo}}$, peaking near the true lower body. But $\beta_{1,\mathrm{lo}}^k$ is a near-prior-distributed random draw (because view 1 has no information about it). The probability that this random draw lands within view 2's tight peak is the curse-of-dimensionality factor from 16.1.

So **every cross-view weight is essentially a draw from a noise process** with a heavy lower-tail penalty: most candidates score near $-\infty$ on the lower term; once in $10^{22}$ might score well. After softmax (tempered or not), weights spread roughly uniformly over the bad candidates and the merged estimate degenerates to an unweighted mean of view 1 and view 2 samples — i.e. half a prior-mean draw per view, no fusion benefit.

### 16.3 The asymmetry that pinpoints the failure

The structure also reveals a useful asymmetry. If a single sample $\beta_1^k$ happened by accident to have $\beta_{1,\mathrm{lo}}^k \approx \beta_{\mathrm{lo}}^{\mathrm{true}}$ (the true lower body), then:

- Under view 2's flow, that sample would score very high (view 2 likes accurate lower).
- Under view 1's own flow, that sample is just one of many equally-likely draws (view 1 doesn't care).

So **a "good" sample is identifiable post hoc by its high cross-view weight, but you have no mechanism to *generate* such a sample with reasonable probability** — view 1 would need to know view 2's information at sampling time. Per-view-proposal IS does not let you do that.

This is exactly why `merge_params_nf_langevin` works in this regime: chains walk in $\beta$-space using the gradient of $\log \hat p_1 + \log \hat p_2$, so the chain is *attracted* to the joint mode rather than relying on a lucky proposal draw. `merge_params_nf_gaussian` sidesteps the problem entirely by using per-view variances analytically — no proposal sampling needed.

### 16.4 One-sentence statement

Per-view proposals concentrate on per-view-thin manifolds, but the joint posterior lives at their intersection, and IS weights cannot manufacture samples in regions the proposal never visits.

## 17. Diagnostics that confirm IS coverage failure

The IS-coverage hypothesis predicts specific signatures. Run these to confirm:

1. **ESS does not collapse on a single sample, but stays close to $V \cdot S$ (uniform).** Compute $\mathrm{ESS} = 1 / \sum_k w_k^2$ over the $V \cdot S$ candidates. If ESS is order $V \cdot S$, the IS reweighting is doing nothing — that confirms no candidate is preferred. (If ESS is $\mathcal{O}(1)$, weights are still collapsing despite tempering — a different problem.)
2. **Tempered merged $\approx$ uniform $V \cdot S$ average.** Direct comparison; if equal, tempering has no signal to amplify.
3. **Gaussian precision-weighted merge (`merge_params_nf_gaussian`) does beat oracle-min.** This is the smoking gun. Gaussian merge uses per-view variances directly — it does not draw cross-view candidates and so is immune to the coverage problem. If Gaussian works and tempered does not, the model is fine; the IS recipe is the bottleneck.
4. **Langevin merge (`merge_params_nf_langevin`) does beat oracle-min.** Langevin walks chains toward the joint mode, generating candidates *in* the joint-high-density region rather than reweighting per-view proposals. If it succeeds, this confirms the issue is sampling coverage, not the flow.

## 18. What this says about the model vs. the merge

The flow's per-view marginals are calibrated (samples are diverse where they should be). The model itself is doing the right thing for any per-view-only inference task. What it does **not** do is provide a *joint* proposal — and the cross-view IS recipe in `merge_params_nf_tempered` requires per-view proposals to overlap the joint posterior, which they cannot when views are strongly complementary.

So the right reading of the negative result is:

> **The model is fine; the merge is wrong for this regime.** Cross-view IS via per-view proposals is fundamentally an exploration-by-luck procedure. With well-overlapping (redundant) views it works because per-view proposals automatically cover each other. With complementary views it fails by construction in high $D$.

The codebase already has the right tools for this regime:

- **`merge_params_nf_gaussian`** — uses per-view variances analytically; coverage-free.
- **`merge_params_nf_langevin`** — explicitly samples the joint posterior via MALA; goes to the joint mode rather than reweighting.

If you want IS specifically, you would need a **joint proposal** that covers both halves — e.g. propose from a Gaussian centred on the precision-weighted mean with covariance $\bigl(\sum_j \Lambda_j\bigr)^{-1}$, then weight by $\hat p_{\mathrm{joint}} / q$. That salvages IS at the cost of Gaussianising the proposal, which loses the flow's higher-order structure.

## 19. The one model-side caveat this argument does not rule out

There is one model-side defect the IS-coverage argument does not exclude: **wrong correlation structure between observed and unobserved components**. The marginals can look diverse while the conditional

$$
\hat p\bigl(\beta_{\mathrm{lower}} \,\big|\, \beta_{\mathrm{upper}}, c_{\mathrm{upper-crop}}\bigr)
$$

is wrong. If view 1's flow has spurious correlations (e.g. it has memorised dataset-level statistics that make $\beta_{\mathrm{lower}}$ track $\beta_{\mathrm{upper}}$ in a way that is true on average over training subjects but wrong for any individual subject), candidates from view 1 may systematically miss view 2's lower posterior even more than the marginal-level argument predicts.

Quick check: condition view 1 samples on the GT upper components and inspect the distribution of lower components. If it looks like the prior, the conditional is correctly broad. If it has structure (a tight mode shifted away from GT lower), the flow has learned a spurious correlation and the joint posterior the IS merge is targeting is itself biased.

# How Langevin Fixes the Coverage Failure

The IS failure in section 16 is fundamentally a **sampling problem**: per-view proposals do not cover the joint mode, and reweighting cannot synthesise samples that do not exist. Langevin (MALA in `merge_params_nf_langevin`) sidesteps this by replacing "draw from per-view, reweight" with "walk in $\beta$-space using gradients of the joint log-density."

## 20. The mechanism

Define the unnormalised joint log-target

$$
\log \pi(\beta) \;=\; \sum_{j=1}^{V} \log \hat p\bigl(\beta - \mu_j \,\big|\, c_j\bigr).
$$

MALA at state $\beta^{(t)}$ proposes

$$
\beta' \;=\; \beta^{(t)} \;+\; \tfrac{h}{2}\, M\, \nabla_\beta \log \pi\bigl(\beta^{(t)}\bigr) \;+\; \sqrt{h\, M}\,\xi, \qquad \xi \sim \mathcal{N}(0, I),
$$

then accepts/rejects via Metropolis–Hastings to make $\pi$ the stationary distribution. The drift uses the **gradient of the joint log-density** — local information about *which direction* increases joint likelihood — rather than a sample drawn from any per-view distribution. The preconditioner $M$ (in the codebase: $M = \bigl(\sum_j \Lambda_j\bigr)^{-1}$, the Gaussian joint covariance) matches step sizes to the local geometry.

## 21. Why gradients escape the curse the proposal could not

By linearity of $\nabla$,

$$
\nabla_\beta \log \pi(\beta) \;=\; \sum_{j=1}^{V} \nabla_\beta \log \hat p\bigl(\beta - \mu_j \,\big|\, c_j\bigr).
$$

Each view contributes a gradient. Restrict to the upper/lower decomposition under extreme cropping:

- **Upper components** of $\nabla \log \pi$: gradient from view 1 (tight on upper) sharply pulls $\beta_{\mathrm{up}}$ toward view 1's tight upper mode. Gradient from view 2 (loose on upper) is near zero. **Net pull on upper: pure view-1 information.**
- **Lower components**: symmetric. Gradient from view 2 (tight on lower) sharply pulls $\beta_{\mathrm{lo}}$ toward the true lower. View 1's lower gradient is near zero. **Net pull on lower: pure view-2 information.**

Each view contributes its information **additively in gradient space**, in exactly the directions it has information about. The chain navigates to the joint mode by following these per-view gradients, **without needing any single view's proposal to cover the joint posterior**.

This is why Langevin is dimension-polynomial where IS is dimension-exponential. IS asks the question:

> "Does $\beta$ have high joint likelihood?" — answered by drawing many candidates and hoping one lands near the joint mode. Probability per candidate: $(\sigma_{\mathrm{obs}}/\sigma_{\mathrm{prior}})^{d_{\mathrm{unobs}}}$ — exponentially small.

Langevin asks the dual question:

> "Where in $\beta$-space is the joint likelihood increasing?" — answered by reading off a gradient. Cost: $D$ partial derivatives, no rare-event probability involved.

A 55-D gradient is just 55 numbers, each carrying real signal. There is no exponential decay because we are not estimating a measure of high-likelihood regions; we are computing a direction that points to them.

## 22. Walk-through on the upper/lower scenario

Initialise the chain at $\beta^{(0)} = \mu_1$ (view 1's mean): correct upper, prior-mean lower (badly wrong on lower). Step by step:

1. Compute $\nabla \log \pi$ at $\mu_1$.
   - Upper components of the gradient: small. View 1's flow gradient is near zero at its own mean; view 2's flow has no opinion on upper.
   - Lower components: large, pointing from prior-mean-lower toward view 2's tight true-lower peak (since view 2's flow gradient on lower is sharp).
2. Propose $\beta' = \mu_1 + \tfrac{h}{2} M \nabla \log \pi + \sqrt{hM}\xi$. The dominant move is in the lower subspace, toward the true lower body.
3. Metropolis–Hastings accepts (high probability — proposal goes uphill).
4. Repeat. The chain moves a step closer to the true lower at each iteration. The upper stays approximately fixed (view 1's flow penalises moves away from true upper).

After $T$ steps, the chain sits near the joint mode: correct upper from view 1, correct lower from view 2. The codebase pools the last `tail_frac` (default 0.4) of the trajectory as the posterior-mean estimate. With $V$ chains running in parallel from different inits, mixing is faster and the pool gives a better mean estimate.

## 23. What the codebase does specifically

`merge_params_nf_langevin` (`nf_merging.py:592`) implements MALA with a few tuning choices that matter:

1. **Preconditioner from the Gaussian joint posterior** (`M_diag = 1.0 / precision_j.sum(dim=1)`, line 673). Step sizes scale per-component to match the joint posterior's anisotropy: large steps in directions where the joint is wide, small in directions where it is tight. Without this, the chain mixes badly in the tight directions.
2. **Variance floor** (lines 656–670). Prevents collapsed dimensions in per-view sample variance estimates from giving infinite precision and pathological step sizes — a calibration guardrail.
3. **`init_strategy="gaussian_warm"`** (line 686). Initialises the $V$ chains at the precision-weighted Gaussian merge estimate (already close to the joint mode under Gaussianity), then perturbs by $\sim \sqrt{M}$ to seed exploration. Much closer to the joint mode than view means and reduces burn-in.
4. **Robbins–Monro step-size adaptation** to the optimal-MALA acceptance rate of $0.574$ (lines 737–743). Keeps mixing efficient as the geometry changes during burn-in.

## 24. Why it works on the slab–intersection problem

Recall the geometry from 16.1: view 1's posterior is a thick slab oriented along the unobserved-by-1 subspace; view 2's slab is orthogonal; their intersection is a small ball. IS fills view 1's slab and almost never hits the ball. Langevin starts somewhere — could be inside view 1's slab, could be on its boundary — and **reads the gradient of $\log \pi_2$**, which points from anywhere in view 1's slab *toward* view 2's slab. The chain walks across slabs, not by sampling, but by following the slope.

If view 1's slab and view 2's slab are perpendicular, the gradient field of the sum points toward their intersection from every starting point. Convergence time depends on slab thickness ratios but is polynomial in $D$, not exponential.

## 25. Diagnostic value of running Langevin

Specifically:

- If you turn on Langevin and it **does** beat oracle-min on the extreme-cropping setup, the model is fine — `flow_beta` is producing well-calibrated per-view posteriors with usable gradients, and the only thing wrong with tempered IS was the proposal-coverage failure.
- If Langevin **does not** beat oracle-min either, the diagnosis shifts: the per-view flows themselves are mis-modelling the posterior (wrong gradient → wrong joint mode), which goes back to mode A1 / A3 — a model-side defect rather than a merge-side one.

Running both Langevin and Gaussian merges on the same trained model is the cleanest way to separate "model is bad" from "merge recipe is bad."